In this project we will try to scrape data from reddit using their API. The objective is to load reddit data into a pandas dataframe. In order to achieve this, first we'll import the following libraries.

The documentation for this API can be found [here](https://github.com/reddit/reddit/wiki/API).

In [1]:
import pandas as pd
import urllib.request as ur
import json
import time

We can access the raw json data of any subreddit by adding '.json' to the URL. Using the urllib.request library, we can extract that data and read it in python.

From the documentation, we need to fill out the header using the suggested format.

Example: User-Agent: android:com.example.myredditapp:v1.2.3 (by /u/kemitche)

In [2]:
#Header to be submitted to reddit.
hdr = {'User-Agent': 'codingdisciple:playingwithredditAPI:v1.0 (by /u/ivis_reine)'}

#Link to the subreddit of interest.
url = "https://www.reddit.com/r/grandorder/.json?sort=top&t=all"

#Makes a request object and receive a response.
req = ur.Request(url, headers=hdr)
response = ur.urlopen(req).read()

#Loads the json data into python.
json_data = json.loads(response.decode('utf-8'))

I took a snapshot of the data structure below. It looks like the data is just a bunch of lists and dictionaries. We want reach the part of the dictionary until we see a list. Each item on this list will be a post made on this subreddit.

![reddit_data_structure](https://raw.githubusercontent.com/sengkchu/Personal-Projects/master/datastructure.png)

In [3]:
#The actual data starts.
data = json_data['data']['children']

Each request can only get us 100 posts, we can write a for loop to send 10 requests at 2 second intervals and add the data to the list of posts.

In [4]:
for i in range(10):
    #reddit only accepts one request every 2 seconds, adds a delay between each loop
    time.sleep(2)
    last = data[-1]['data']['name']
    url = 'https://www.reddit.com/r/grandorder/.json?sort=top&t=all&limit=100&after=%s' % last
    req = ur.Request(url, headers=hdr)
    text_data = ur.urlopen(req).read()
    datatemp = json.loads(text_data.decode('utf-8'))
    data += datatemp['data']['children']
    print(str(len(data))+" posts loaded")
    

127 posts loaded
227 posts loaded
327 posts loaded
427 posts loaded
527 posts loaded
627 posts loaded
727 posts loaded
827 posts loaded
927 posts loaded
927 posts loaded


We've assigned all the posts to a list with the variable named 'data'. In order to begin constructing our pandas dataframe, we need a list of column names. Each post consists of a dictionary, we can simply loop through this dictionary and extract the column names.

In [5]:
#Create a list of column name strings to be used to create our pandas dataframe
data_names = [value for value in data[0]['data']]
print(data_names)

['edited', 'subreddit_name_prefixed', 'domain', 'is_reddit_media_domain', 'spoiler', 'selftext_html', 'selftext', 'url', 'brand_safe', 'mod_reason_by', 'stickied', 'locked', 'id', 'name', 'ups', 'preview', 'suggested_sort', 'view_count', 'link_flair_text', 'thumbnail', 'link_flair_css_class', 'report_reasons', 'downs', 'likes', 'banned_by', 'thumbnail_width', 'author', 'clicked', 'num_reports', 'hidden', 'contest_mode', 'mod_reports', 'visited', 'whitelist_status', 'created_utc', 'title', 'approved_at_utc', 'can_gild', 'secure_media_embed', 'quarantine', 'secure_media', 'parent_whitelist_status', 'is_video', 'banned_at_utc', 'mod_note', 'thumbnail_height', 'is_crosspostable', 'permalink', 'hide_score', 'pinned', 'post_hint', 'num_comments', 'created', 'user_reports', 'archived', 'can_mod_post', 'gilded', 'over_18', 'media', 'num_crossposts', 'mod_reason_title', 'subreddit', 'removal_reason', 'is_self', 'subreddit_type', 'author_flair_text', 'media_embed', 'saved', 'score', 'distinguish

In order to build a dataframe using the pd.DataFrame() function, we will need a list of dictionaries. 

We can loop through each element in 'data', using each column name as a key to the dictionary, then accessing the corresponding value with that key. If we come across a post that has

In [6]:
#Create a list of dictionaries to be loaded into a pandas dataframe
df_loadinglist = []
for i in range(0, len(data)):
    dictionary = {}
    for names in data_names:
        try:
            dictionary[str(names)] = data[i]['data'][str(names)]
        except:
            dictionary[str(names)] = 'None'
    df_loadinglist.append(dictionary)
df=pd.DataFrame(df_loadinglist)

In [8]:
df.shape

(927, 73)

In [9]:
df.tail()

,approved_at_utc,approved_by,archived,author,author_flair_css_class,author_flair_text,banned_at_utc,banned_by,brand_safe,can_gild,...,thumbnail,thumbnail_height,thumbnail_width,title,ups,url,user_reports,view_count,visited,whitelist_status
922,None,None,False,ArcueidAwoo,9-solomonaf,H-help me~,None,None,True,False,...,https://a.thumbs.redditmedia.com/-9ZLqglo3aAO6...,105.0,140.0,Seal Thirteen... Decision Start!,151,https://www.youtube.com/watch?v=X962oII_Oo8,[],None,False,all_ads
923,None,None,False,SeijoVangelta,3-mordred1,Tomboy Fetish,None,None,True,False,...,self,NaN,NaN,Heroic Spirits summoned in the Holy Grail Wars...,120,https://www.reddit.com/r/grandorder/comments/7...,[],None,False,all_ads
924,None,None,False,ArchadianJudge,3-caren,Caren &amp; Saber Alter's Husband,None,None,True,False,...,self,NaN,NaN,Is your game running fine? Japanese players (a...,46,https://www.reddit.com/r/grandorder/comments/7...,[],None,False,all_ads
925,None,None,False,AutoModerator,bbmodflair,♥ HAIIIII ♥,None,None,True,False,...,self,NaN,NaN,[Fanart Megathread] BBs Fanart Folder!,25,https://www.reddit.com/r/grandorder/comments/7...,[],None,False,all_ads
926,None,None,False,GraveRobberJ,5-nobu2,,None,None,True,False,...,https://b.thumbs.redditmedia.com/fPcmS4bwyULm9...,140.0,140.0,Kiara receives a massage,85,https://twitter.com/_JB_K/status/9503128905562...,[],None,False,all_ads


Now that we have a pandas dataframe, we can do simple analysis on the reddit posts. For example, we can write a function to find the most common words used in the last 925 posts.

In [10]:
#Counts each word and return a pandas series

def word_count(df, column):
    dic={}
    for idx, row in df.iterrows():
        split = row[column].lower().split(" ")
        for word in split:
            if word in dic:
                dic[word] += 1
            else:
                dic[word] = 1
    dictionary = pd.Series(dic)
    dictionary = dictionary.sort_values(ascending=False)
    return dictionary

top_counts = word_count(df, "selftext")
top_counts[0:5]

the    2490
to     1870
       1609
of     1407
and    1376
dtype: int64

The results are not too surprising, common english words showed up the most.

That is it for now! We've achieved our goal of turning json data into a pandas dataframe.

---

#### Learning Summary

Concepts explored: lists, dictionaries, for loops, API, data structures, JSON

The files for this project can be found in my [GitHub repository](https://github.com/sengkchu/Personal-Projects).